In [1]:
# Install required packages
!pip install langchain-core langchain --quiet
print("Installation complete.")

Installation complete.


In [2]:
# langchain_core is the correct import path for LangChain v0.2+
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
print("Imports successful.")

Imports successful.


## Task 1: Replace Hardcoded Prompts

In [3]:
# Reusable PromptTemplate — replaces the original f-string function
explain_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

# Test the template
output = explain_template.format(topic="Machine Learning")
print("Task 1 Output:")
print(" ", output)

Task 1 Output:
  Explain Machine Learning in simple terms for beginners


## Task 2: Multi-Input Prompt System

In [4]:
# Multi-input template
multi_input_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# Required test cases
test_cases = [
    {"topic": "AI",            "audience": "beginners", "tone": "friendly"},
    {"topic": "Python",        "audience": "kids",      "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers", "tone": "technical"},
]

print("Task 2 Output:")
for i, case in enumerate(test_cases, 1):
    formatted_prompt = multi_input_template.format(**case)
    print("  Test " + str(i) + ": " + formatted_prompt)


Task 2 Output:
  Test 1: Explain AI for beginners in a friendly tone
  Test 2: Explain Python for kids in a fun tone
  Test 3: Explain Deep Learning for engineers in a technical tone


## Task 3: Prompt Variations Engine

In [5]:
# Three variation templates — logic stays separate from template strings
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step"
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 interview questions about {topic}"
)

storytelling_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story"
)

# Store templates in a dict for easy iteration
variation_templates = {
    "Teaching":     teaching_template,
    "Interview":    interview_template,
    "Storytelling": storytelling_template,
}

topic = "Machine Learning"
print("Task 3 Output:")
for style, template in variation_templates.items():
    formatted_prompt = template.format(topic=topic)
    print("  [" + style + "] → " + formatted_prompt)


Task 3 Output:
  [Teaching] → Explain Machine Learning clearly step by step
  [Interview] → Ask 3 interview questions about Machine Learning
  [Storytelling] → Explain Machine Learning as a story


## Task 4: ChatPromptTemplate System

In [8]:
# Role descriptions kept separate from the template
ROLE_DESCRIPTIONS = {
    "teacher":     "You are an experienced teacher who explains concepts clearly and patiently.",
    "interviewer": "You are a technical interviewer assessing candidate knowledge rigorously.",
    "motivator":   "You are an enthusiastic motivator who inspires learners to keep going.",
}

# Single reusable ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages([
    ("system", "{system_description}"),
    ("human",  "Tell me about {topic}."),
])

def build_chat_prompt(role: str, topic: str):
    system_description = ROLE_DESCRIPTIONS.get(role, "You are a helpful assistant.")
    messages = chat_template.format_messages(
        system_description=system_description,
        topic=topic
    )
    return messages

# Example: role = "teacher", topic = "Neural Networks"
messages = build_chat_prompt(role="teacher", topic="Neural Networks")

print("Task 4 Output:")
for msg in messages:
    print("  [" + msg.type.upper() + "]: " + msg.content)

# Test all three roles
print("All Roles for topic = 'Neural Networks':")
for role in ROLE_DESCRIPTIONS:
    msgs = build_chat_prompt(role, "Neural Networks")
    print("  [" + role.upper() + "] System → " + msgs[0].content[:60] + "...")

Task 4 Output:
  [SYSTEM]: You are an experienced teacher who explains concepts clearly and patiently.
  [HUMAN]: Tell me about Neural Networks.
All Roles for topic = 'Neural Networks':
  [TEACHER] System → You are an experienced teacher who explains concepts clearly...
  [INTERVIEWER] System → You are a technical interviewer assessing candidate knowledg...
  [MOTIVATOR] System → You are an enthusiastic motivator who inspires learners to k...


## Task 5: Input Validation Layer

In [10]:
# Allowed values — single source of truth
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES     = ["formal", "casual", "fun"]

def validate_inputs(audience: str, tone: str) -> tuple:
    validated_audience = audience if audience in VALID_AUDIENCES else "beginner"
    validated_tone     = tone     if tone     in VALID_TONES     else "casual"

    if audience not in VALID_AUDIENCES:
        print("  Invalid audience \"" + audience + "\" → defaulting to \"beginner\"")
    if tone not in VALID_TONES:
        print("  Invalid tone \"" + tone + "\" → defaulting to \"casual\"")

    return validated_audience, validated_tone

print("Task 5 Output – Validation Tests:")
print("  Valid input:        " + str(validate_inputs("beginner", "formal")))
print("  Invalid audience:   " + str(validate_inputs("novice", "fun")))
print("  Invalid tone:       " + str(validate_inputs("expert", "angry")))
print("  Both invalid:       " + str(validate_inputs("alien", "sleepy")))

Task 5 Output – Validation Tests:
  Valid input:        ('beginner', 'formal')
  Invalid audience "novice" → defaulting to "beginner"
  Invalid audience:   ('beginner', 'fun')
  Invalid tone "angry" → defaulting to "casual"
  Invalid tone:       ('expert', 'casual')
  Invalid audience "alien" → defaulting to "beginner"
  Invalid tone "sleepy" → defaulting to "casual"
  Both invalid:       ('beginner', 'casual')


## Task 6: Prompt Generator App (15%)

`generate_prompt(topic, audience, tone, style)` runs the full pipeline: validate → select template → format → return.

In [11]:
# Style templates — each uses all three input variables
STYLE_TEMPLATES = {
    "teaching": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} teaching style, step by step"
    ),
    "interview": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Ask 3 {tone} interview questions about {topic} suited for an {audience}"
    ),
    "storytelling": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} storytelling style"
    ),
}

def generate_prompt(topic: str, audience: str, tone: str, style: str) -> str:
    audience, tone = validate_inputs(audience, tone)

    if style not in STYLE_TEMPLATES:
        raise ValueError("Style not supported. Choose from: " + str(list(STYLE_TEMPLATES.keys())))
    template = STYLE_TEMPLATES[style]

    return template.format(topic=topic, audience=audience, tone=tone)

print("Task 6 Output:")
print("  " + generate_prompt("Neural Networks", "beginner", "fun", "storytelling"))
print("  " + generate_prompt("Pandas", "intermediate", "casual", "teaching"))
print("  " + generate_prompt("SQL", "expert", "formal", "interview"))


Task 6 Output:
  Explain Neural Networks for beginner in a fun storytelling style
  Explain Pandas for intermediate in a casual teaching style, step by step
  Ask 3 formal interview questions about SQL suited for an expert


## Task 7: Template Reusability Test

In [12]:
# ONE template used for all 5 runs
reusable_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# 5 varied input sets
inputs = [
    {"topic": "Neural Networks",        "audience": "beginners",     "tone": "friendly"},
    {"topic": "Data Structures",        "audience": "students",      "tone": "formal"},
    {"topic": "Reinforcement Learning", "audience": "researchers",   "tone": "technical"},
    {"topic": "Python Decorators",      "audience": "kids",          "tone": "fun"},
    {"topic": "Cloud Computing",        "audience": "business team", "tone": "casual"},
]

print("Task 7 – Reusability Test (ONE template → 5 different outputs):")
for i, inp in enumerate(inputs, 1):
    formatted_prompt = reusable_template.format(**inp)
    print("  Output " + str(i) + ": " + formatted_prompt)


Task 7 – Reusability Test (ONE template → 5 different outputs):
  Output 1: Explain Neural Networks for beginners in a friendly tone
  Output 2: Explain Data Structures for students in a formal tone
  Output 3: Explain Reinforcement Learning for researchers in a technical tone
  Output 4: Explain Python Decorators for kids in a fun tone
  Output 5: Explain Cloud Computing for business team in a casual tone


## Full Pipeline Demo

```
User Input → Validation → Prompt Template → Dynamic Prompt Generation → Output
```

In [13]:
print("=" * 65)
print("  MINI PROMPT ENGINE — Full Pipeline")
print("=" * 65)

user_topic    = "Transformers"
user_audience = "intermediate"
user_tone     = "casual"
user_style    = "teaching"

print("  Input  → topic: "" + user_topic + "" | audience: "" + user_audience + "" | tone: "" + user_tone + "" | style: "" + user_style + """)
result = generate_prompt(user_topic, user_audience, user_tone, user_style)
print("  Output → " + result)
print("=" * 65)


  MINI PROMPT ENGINE — Full Pipeline
  Input  → topic:  + user_topic +  | audience:  + user_audience +  | tone:  + user_tone +  | style:  + user_style + 
  Output → Explain Transformers for intermediate in a casual teaching style, step by step
